<!-- NB_DOC_INTRO_V1 -->
# MLOps — Tracking et versioning : MLflow + DVC + Weights & Biases

> 📚 **Doc thematique** : [docs/08_MLOPS.md](docs/08_MLOPS.md) (MLOps)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**Sans tracking, ML = artisanat irreproductible.** Question typique : "Pourquoi mon modele d'il y a 3 mois faisait 87% et celui d'aujourd'hui 82% ? Quelles donnees ? Quels hyperparametres ? Quelle archi ?". Le tracking repond a ca.

3 piliers du tracking ML :

| Pilier | Objet | Outil de reference |
|---|---|---|
| **Code** | Versions du code | Git |
| **Data** | Versions des datasets / modeles | **DVC** (Data Version Control) |
| **Experiences** | Params, metrics, artefacts | **MLflow**, **Weights & Biases**, Neptune |

Ce notebook execute **MLflow en local** (le standard open-source) + presente DVC et W&B avec leur API. Couvre :
- Tracking d'experiences (params, metrics, artefacts)
- Model Registry (stages : Staging → Production)
- Auto-logging (sklearn, XGBoost, PyTorch Lightning)
- Comparaison MLflow vs W&B vs Neptune vs ClearML
- Structure de projet recommandee (Cookiecutter Data Science)

Versions : `mlflow >= 2.10`.

## Plan

1. Installation et concepts
2. **MLflow Tracking** — run, log_param, log_metric, log_model
3. **MLflow autolog** (sklearn, xgboost, pytorch)
4. **Model Registry** — versionning et stages
5. **DVC** — versionning des donnees (presentation + commandes)
6. **Weights & Biases** — alternative SaaS moderne (code)
7. **Cookiecutter Data Science** — structure projet recommandee
8. Comparatif des outils
9. Pieges et anti-patterns
10. References


## 1. Installation et concepts

### Concepts MLflow
- **Experiment** : groupe logique de runs (ex: "churn_v2")
- **Run** : 1 entrainement, identifie par `run_id`
- **Param** : hyperparametre (immutable apres set)
- **Metric** : valeur numerique, peut etre logged plusieurs fois (courbe)
- **Artifact** : fichier (modele.pkl, plot.png, requirements.txt)
- **Tag** : metadata libre


In [ ]:
# pip install -q mlflow scikit-learn
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

print(f"MLflow version : {mlflow.__version__}")

SEED = 42
X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

### 1.1 Configurer le tracking URI

Par defaut MLflow loggue dans `./mlruns/`. On peut pointer vers un serveur distant (`http://mlflow-server:5000`) ou un backend Postgres.


In [ ]:
import tempfile, os

# Pour la demo, on logue dans un repertoire temporaire isole
demo_dir = tempfile.mkdtemp(prefix="mlflow_demo_")
mlflow.set_tracking_uri(f"file:///{demo_dir.replace(os.sep, '/')}")
mlflow.set_experiment("breast_cancer_demo")
print(f"Tracking URI : {mlflow.get_tracking_uri()}")
print(f"Experiment   : {mlflow.get_experiment_by_name('breast_cancer_demo').name}")

## 2. MLflow Tracking — workflow standard


In [ ]:
# === Run 1 : LogReg baseline ===
with mlflow.start_run(run_name="logreg_baseline") as run:
    # Hyperparams (immutables)
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("C", 1.0)

    # Fit
    model = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED)
    model.fit(X_tr, y_tr)

    # Metriques (peuvent etre loggees plusieurs fois pour faire des courbes)
    proba = model.predict_proba(X_te)[:, 1]
    pred = (proba >= 0.5).astype(int)
    mlflow.log_metric("test_accuracy", accuracy_score(y_te, pred))
    mlflow.log_metric("test_f1", f1_score(y_te, pred))
    mlflow.log_metric("test_auc", roc_auc_score(y_te, proba))

    # Modele
    mlflow.sklearn.log_model(model, "model")

    # Tag
    mlflow.set_tag("dev", "florian")
    mlflow.set_tag("dataset", "breast_cancer_sklearn")

    run_id_1 = run.info.run_id
    print(f"Run 1 (LogReg) : id={run_id_1[:8]}, AUC={roc_auc_score(y_te, proba):.4f}")

In [ ]:
# === Run 2 : RandomForest avec differents n_estimators (courbe) ===
with mlflow.start_run(run_name="rf_tuning") as run:
    mlflow.log_param("model", "RandomForestClassifier")

    best_auc = 0
    best_n = None
    for n in [10, 50, 100, 200]:
        m = RandomForestClassifier(n_estimators=n, random_state=SEED, n_jobs=-1)
        m.fit(X_tr, y_tr)
        auc = roc_auc_score(y_te, m.predict_proba(X_te)[:, 1])
        # Log metric avec step = courbe
        mlflow.log_metric("test_auc", auc, step=n)
        print(f"  n_estimators={n}: AUC={auc:.4f}")
        if auc > best_auc:
            best_auc = auc; best_n = n; best_model = m

    mlflow.log_param("best_n_estimators", best_n)
    mlflow.log_metric("best_auc", best_auc)
    mlflow.sklearn.log_model(best_model, "model")
    run_id_2 = run.info.run_id
    print(f"Run 2 (RF) : id={run_id_2[:8]}, best AUC={best_auc:.4f}")

In [ ]:
# === Run 3 : avec input_example + signature ===
from mlflow.models.signature import infer_signature

with mlflow.start_run(run_name="rf_with_signature") as run:
    m = RandomForestClassifier(n_estimators=100, random_state=SEED).fit(X_tr, y_tr)

    # Signature = schema input/output (pour deploiement)
    signature = infer_signature(X_tr, m.predict(X_tr))
    input_example = X_tr[:5]

    mlflow.sklearn.log_model(
        m, "model",
        signature=signature,
        input_example=input_example,
    )
    print(f"Run 3 (signed) : id={run.info.run_id[:8]}")

### 2.1 Lister les runs et les comparer


In [ ]:
# Recuperer tous les runs de l'experiment
exp = mlflow.get_experiment_by_name("breast_cancer_demo")
runs_df = mlflow.search_runs(experiment_ids=[exp.experiment_id])
print(f"Nb runs : {len(runs_df)}")
print(f"\nColumns : {[c for c in runs_df.columns if not c.startswith('tags.')][:10]}")
print(f"\nRuns :")
runs_df[["run_id", "tags.mlflow.runName", "metrics.test_auc", "metrics.best_auc", "status"]].head()

### 2.2 Charger un modele depuis MLflow


In [ ]:
# Charger le modele du run 1 (LogReg)
model_uri = f"runs:/{run_id_1}/model"
loaded = mlflow.sklearn.load_model(model_uri)

# Predire
acc = accuracy_score(y_te, loaded.predict(X_te))
print(f"Modele charge depuis MLflow, accuracy : {acc:.4f}")

## 3. MLflow autolog

Au lieu de logger manuellement, **`mlflow.<lib>.autolog()`** loggue **tout automatiquement** : params, metrics standards, modele, training plots.

Disponible pour : `sklearn`, `xgboost`, `lightgbm`, `pytorch`, `keras`, `tensorflow`, `spark`, `fastai`, `transformers`, `pyspark`.


In [ ]:
mlflow.sklearn.autolog(log_models=True, silent=True)

with mlflow.start_run(run_name="rf_autolog") as run:
    m = RandomForestClassifier(n_estimators=50, max_depth=8, random_state=SEED).fit(X_tr, y_tr)
    # Tout est logge AUTOMATIQUEMENT : params, metrics (training_score), model

print(f"Run autolog : {run.info.run_id[:8]}")

# Voir ce qui a ete logge
loaded_run = mlflow.get_run(run.info.run_id)
print(f"  Params auto: {dict(list(loaded_run.data.params.items())[:5])}")
print(f"  Metrics auto: {dict(loaded_run.data.metrics)}")

# Important : disable autolog si tu veux logger manuellement apres
mlflow.sklearn.autolog(disable=True)

## 4. Model Registry — versionning et stages

Le **Model Registry** permet d'organiser les modeles avec :
- **Name** : un nom (ex: "churn-predictor")
- **Version** : auto-incremente a chaque enregistrement
- **Stage** : `None` / `Staging` / `Production` / `Archived`
- **Description**, tags, lineage


In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
model_name = "demo_breast_cancer_clf"

# Enregistrer le modele du run 2 (RF) dans le registry
result = mlflow.register_model(
    model_uri=f"runs:/{run_id_2}/model",
    name=model_name,
)
print(f"Modele enregistre : {result.name} v{result.version}")

# Promouvoir vers Staging (transition_model_version_stage est deprecate depuis MLflow 2+)
# Alternative moderne : set_registered_model_alias
client.set_registered_model_alias(name=model_name, alias="staging", version=result.version)
print(f"  → tag 'staging' assigne a v{result.version}")

# Liste des versions
versions = client.search_model_versions(f"name='{model_name}'")
for v in versions:
    print(f"  v{v.version} : aliases={v.aliases}, run_id={v.run_id[:8]}")

In [ ]:
# Charger via alias (pattern moderne, recommande)
model_uri = f"models:/{model_name}@staging"
loaded = mlflow.sklearn.load_model(model_uri)
print(f"Loaded via alias 'staging' : {type(loaded).__name__}")

## 5. DVC (Data Version Control) — versionner les donnees

DVC = **"Git pour les gros fichiers"** + pipeline declaratif. Les commandes :

```bash
# Setup (une fois)
pip install dvc dvc-s3                # ou dvc-gs, dvc-azure, dvc-ssh
dvc init                              # dans un repo Git existant
git commit -m "Init DVC"

# Tracker un fichier (data, modele)
dvc add data/raw/train.csv            # → cree train.csv.dvc (small file → Git)
                                       # → le vrai file va dans .dvc/cache

# Push vers stockage remote
dvc remote add -d storage s3://my-bucket/dvc-cache
dvc push                              # envoie les vrais fichiers vers S3

# Sur une autre machine
git clone <repo>
dvc pull                              # telecharge les fichiers depuis S3
```

### Pipeline DVC (`dvc.yaml`)

```yaml
stages:
  prepare:
    cmd: python src/prepare.py data/raw data/processed
    deps: [src/prepare.py, data/raw]
    outs: [data/processed]

  train:
    cmd: python src/train.py
    deps: [src/train.py, data/processed]
    params: [train.lr, train.epochs]
    outs: [models/model.pkl]
    metrics:
      - metrics/train.json:
          cache: false                # ne pas DVC-track les metrics

  evaluate:
    cmd: python src/evaluate.py
    deps: [src/evaluate.py, models/model.pkl, data/processed]
    metrics:
      - metrics/eval.json:
          cache: false
```

Usage :
```bash
dvc repro              # rejoue le pipeline en intelligent (skip stages a jour)
dvc dag                # affiche le DAG
dvc metrics show -a    # compare metriques entre branches Git
dvc exp run            # experiment sans Git commit
```


## 6. Weights & Biases — alternative SaaS moderne

W&B = MLflow-like + UI premium + Sweeps (hyperparam) + Reports collaboratifs. SaaS payant a partir d'un certain volume, free tier genereux pour solo.

```python
# pip install -q wandb
# wandb login          # entrer la cle API (gratuite sur wandb.ai)

import wandb

wandb.init(
    project="my-project",
    name="rf-baseline",
    config={
        "model": "RandomForest",
        "n_estimators": 100,
        "max_depth": 10,
    },
    tags=["baseline", "v1"],
)
config = wandb.config

# Entrainer
model = RandomForestClassifier(n_estimators=config.n_estimators,
                                max_depth=config.max_depth).fit(X_tr, y_tr)
proba = model.predict_proba(X_te)[:, 1]
auc = roc_auc_score(y_te, proba)

# Logger
wandb.log({"test_auc": auc, "test_acc": accuracy_score(y_te, (proba >= 0.5).astype(int))})

# Artifact
import joblib
joblib.dump(model, "model.pkl")
artifact = wandb.Artifact("rf-model", type="model")
artifact.add_file("model.pkl")
wandb.log_artifact(artifact)

wandb.finish()
```

### W&B Sweeps (hyperparameter search)
```yaml
# sweep.yaml
method: bayes
metric: {name: test_auc, goal: maximize}
parameters:
  n_estimators: {distribution: int_uniform, min: 10, max: 500}
  max_depth:    {values: [3, 5, 10, 20]}
```
```bash
wandb sweep sweep.yaml     # → renvoie sweep_id
wandb agent <user>/<proj>/<sweep_id> --count 50
```


## 7. Cookiecutter Data Science — structure projet

```bash
pip install cookiecutter
cookiecutter https://github.com/drivendata/cookiecutter-data-science
```

Genere :
```
project/
├── data/
│   ├── raw/                # JAMAIS modifie, sous DVC
│   ├── interim/            # transformations intermediaires
│   ├── processed/          # donnees finales prêtes pour ML
│   └── external/
├── models/                 # modeles serialises (sous DVC)
├── notebooks/              # exploration (numerotes 01-, 02-...)
├── references/             # docs, papers
├── reports/                # rapports + figures
│   └── figures/
├── src/                    # code Python
│   ├── data/               # ingestion
│   ├── features/           # feature engineering
│   ├── models/             # training, prediction
│   └── visualization/
├── tests/
├── .dvc/
├── dvc.yaml                # pipeline
├── params.yaml             # hyperparametres
├── Makefile                # taches standardisees
├── requirements.txt        # ou pyproject.toml + uv
└── README.md
```


## 8. Comparatif des outils de tracking

| Critere | **MLflow** | **W&B** | **Neptune** | **ClearML** |
|---|---|---|---|---|
| Open source | ✅ | UI fermee | ❌ (cloud) | ✅ |
| Self-hosted facile | ✅ | Payant | Payant | ✅ |
| UI | Correct | **Excellente** | Bonne | Bonne |
| Logging | Manuel + autolog | Auto + collab | Bon | Bon |
| Hyperparam sweep | Via Optuna | **Sweeps natif** | Via Optuna | Native |
| Model registry | ✅ | ✅ | ✅ | ✅ |
| Reports collab | Limite | ✅ Excellent | ✅ | ✅ |
| Tarification | Gratuit (self-host) | Free tier + payant | Payant | Open source |

### Recommandation 2025 par cas

| Cas | Recommandation |
|---|---|
| Open source / on-prem | **MLflow** |
| SaaS confortable, equipe data | **W&B** |
| Recherche academique | **Neptune** |
| DevOps existant, scale | **ClearML** |
| Demarrer simple | **MLflow** local + Git + DVC |


## 9. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correction |
|---|---|
| Pousser des fichiers > 100 MB dans Git | DVC pour data, MLflow/W&B pour modeles |
| Logger manuellement HP dans le code | `autolog()` ou `MlflowClient` |
| 1 seul run sans tag → impossible a retrouver | `tags={"experiment": "v2", "dev": "florian"}` |
| Logger TOUT en autolog (overkill, log gigantesque) | Selectif : `autolog(log_models=False)` |
| Pas de signature/input_example | Toujours pour deploiement |
| Promouvoir directement en Production | Pipeline : Staging → tests → Production |
| Pas de tracking URI partage | `MLFLOW_TRACKING_URI` en env var partagee equipe |
| Confondre param (immutable) et metric (peut evoluer) | param = HP setting initial, metric = mesure (loss curve) |
| Pas versionner `requirements.txt` / `uv.lock` | Sans, reproductibilite cassee |


## 10. References

### Documentation officielle
- **MLflow** : https://mlflow.org/docs/latest/index.html
- **DVC** : https://dvc.org/doc
- **Weights & Biases** : https://docs.wandb.ai/
- **Neptune** : https://docs.neptune.ai/
- **ClearML** : https://clear.ml/docs/
- **Cookiecutter Data Science** : https://drivendata.github.io/cookiecutter-data-science/

### Voir aussi (collection)
- [ML_MLFlow_Bench.ipynb](./ML_MLFlow_Bench.ipynb) — MLflow sur bench multi-modeles
- [MLOps_Pipelines_Airflow_Prefect.ipynb](./MLOps_Pipelines_Airflow_Prefect.ipynb) — orchestration
- [MLOps_Drift_Monitoring.ipynb](./MLOps_Drift_Monitoring.ipynb) — monitoring post-deploiement
- [SoftEng_Tests_Quality_ML.ipynb](./SoftEng_Tests_Quality_ML.ipynb) — tests, lint, mypy
